# Asklytics — Developer Code Walkthrough

## 1. Core Agent Module (`core_agent.py`)

This module manages connection initialization to the Google Gemini API using the `google-genai` SDK and implements the **Self-Healing execution engine**.

### Cell [1.1]: Engine Initialization & Code Generator

**Explanation:**  
We define a class `SelfHealingAnalystEngine` to coordinate Gemini client requests. The model uses a detailed `system_instruction` to command the AI to output **only raw Python code** without markdown wrapper lines (like \`\`\`python). The system enforces strict output contracts: any textual insight must be saved in `text_insight` and any Plotly visualization must be assigned to `fig`.



In [ ]:
import os
import pandas as pd
from google import genai

class SelfHealingAnalystEngine:
    def __init__(self):
        # Initialize the Google GenAI client (Gemini 2.5 Flash model)
        self.client = genai.Client()
        self.model_name = "gemini-2.5-flash"

    def generate_analysis_code(self, prompt_context: str, error_message: str = None) -> str:
        """Asks Gemini to write clean analytical code based on available variables."""
        system_instruction = (
            "You are an elite Data Scientist and expert Python developer. Your job is to output ONLY executable Python code.\n"
            "Do NOT wrap code blocks inside markdown text notation like ```python. Return ONLY raw code lines.\n\n"
            "STRICT ARCHITECTURAL EXECUTION RULES:\n"
            "1. You must save your verbal response/insight in a string variable named 'text_insight'.\n"
            "2. If the user query implies an observation, trend, distribution, comparison or explicitly asks for a chart, "
            "you MUST build a Plotly Express chart asset assigned to a variable named 'fig'.\n"
            "3. Example template format:\n"
            "   text_insight = 'The leading categories are...'\n"
            "   fig = px.bar(df, x='AnyColumnFound', y='AnyNumericColumn')"
        )
        
        user_prompt = f"Dataset Environment Blueprint:\n{prompt_context}\n\n"
        if error_message:
            user_prompt += f"⚠️ RUNTIME WARNING: CRITICAL PREVIOUS SCRIPT ATTEMPT FAILED WITH ERROR:\n{error_message}\nRewrite the syntax code to fix this exception safely.\n"
        
        user_prompt += "Generate pristine Python analysis code lines execution block:"
        
        # Call the Google Gemini API
        response = self.client.models.generate_content(
            model=self.model_name,
            contents=user_prompt,
            config=genai.types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.1
            )
        )
        
        return response.text.replace("```python", "").replace("```", "").strip()



### Cell [1.2]: Safe Execution & Self-Healing Loop

**Explanation:**  
The `execute_safely` loop is the core of the self-healing capability. It attempts to run the generated code block using Python's native `exec()` function. If the execution fails (for example, if the AI attempts to use a column name with a different case or key mismatch), the `except` block catches the exception, captures the traceback error logs, and immediately feeds them back into the generator to write a corrected version of the script. It retries this self-correction up to 3 times before raising an error.



In [ ]:
    def execute_safely(self, code_str: str, global_vars: dict, max_retries=3):
        """Monitors and catches runtime errors during evaluation loops to achieve self-healing targets."""
        for attempt in range(max_retries):
            try:
                local_scope_vars = {}
                exec(code_str, global_vars, local_scope_vars)
                return local_scope_vars
            except Exception as runtime_error:
                error_log_trace = f"{type(runtime_error).__name__}: {str(runtime_error)}"
                if attempt == max_retries - 1:
                    raise Exception(f"Self-Correction processing threshold limits breached: {error_log_trace}")
                
                # Regenerate script with explicit error traceback context
                code_str = self.generate_analysis_code(prompt_context=code_str, error_message=error_log_trace)



---

## 2. Ingestion Utilities Module (`file_parsers.py`)

This utility handles reading tabular files (CSV, Excel) and extracting raw string blocks from unstructured files (PDF, Word).

### Cell [2.1]: Structured and Unstructured File Parsers

**Explanation:**  
- `parse_structured_file`: Loads datasets into Pandas. To keep data clean for downstream analytics, it deletes rows that are entirely empty (`dropna`), filters out system columns like `"Unnamed"`, and strips leading/trailing whitespaces from the column headers to make variable naming predictable.
- `parse_unstructured_file`: Decodes PDFs (using page-by-page extraction via `PyPDF`) or DOCX (via paragraph parsing) into single string streams.



In [ ]:
import pandas as pd
from pypdf import PdfReader
from docx import Document
import io

def parse_structured_file(uploaded_file):
    """Parses and sanitizes any Excel sheet or CSV formatting instantly."""
    if uploaded_file.name.endswith('.csv'):
        df = pd.read_csv(uploaded_file)
    else:
        df = pd.read_excel(uploaded_file)
    
    # Drop rows that are entirely empty or columns with no header names
    df = df.dropna(how='all').loc[:, ~df.columns.str.contains('^Unnamed', case=False, na=False)]
    
    # Strip whitespace characters from headers
    df.columns = df.columns.str.strip()
    return df

def parse_unstructured_file(uploaded_file):
    """Extracts raw text string blocks out of PDFs and Word documents."""
    text_content = ""
    
    if uploaded_file.name.endswith('.pdf'):
        pdf_reader = PdfReader(io.BytesIO(uploaded_file.read()))
        for page in pdf_reader.pages:
            extracted = page.extract_text()
            if extracted:
                text_content += extracted + "\n"
                
    elif uploaded_file.name.endswith('.docx'):
        doc = Document(io.BytesIO(uploaded_file.read()))
        for para in doc.paragraphs:
            text_content += para.text + "\n"
            
    return text_content.strip()



---

## 3. Web Controller Router (`app.py`)

This file runs the FastAPI backend engine, handles file ingestions, structures AI prompts, parses responses, and exports analytics data.

### Cell [3.1]: Middleware & Query Endpoint

**Explanation:**  
We configure `CORSMiddleware` to allow asynchronous browser connections to cross port boundaries. The `/api/analyze` route accepts files, loads them into memory, serializes tabular headers to construct prompt boundaries, calls `SelfHealingAnalystEngine` to execute the generated code, and converts Plotly charts to clean client-ready JSON using `PlotlyJSONEncoder`.



In [ ]:
import os
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import pandas as pd
import io
from file_parsers import parse_structured_file, parse_unstructured_file
from core_agent import SelfHealingAnalystEngine

app = FastAPI()

# Enable cross-origin communication for frontend client-side code
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

engine = SelfHealingAnalystEngine()

@app.post("/api/analyze")
async def analyze_data(
    files: list[UploadFile] = File(...),
    query: str = Form(...)
):
    try:
        context_parts = []
        execution_scope = {"pd": pd}
        has_tabular = False
        import plotly.express as px
        
        for idx, file in enumerate(files[:3]):
            file_bytes = await file.read()
            file_name = file.filename
            is_tabular = file_name.endswith(('.csv', '.xlsx'))
            
            if is_tabular:
                has_tabular = True
                if file_name.endswith('.csv'):
                    df = pd.read_csv(io.BytesIO(file_bytes))
                else:
                    df = pd.read_excel(io.BytesIO(file_bytes))
                
                df = df.dropna(how='all').loc[:, ~df.columns.str.contains('^Unnamed', case=False, na=False)]
                df.columns = df.columns.str.strip()
                
                df_var_name = f"df{idx+1}" if len(files) > 1 else "df"
                execution_scope[df_var_name] = df
                
                context_parts.append(
                    f"File {idx+1} ({file_name}) loaded as DataFrame '{df_var_name}'. "
                    f"Columns: {list(df.columns)}. "
                    f"Sample:\n{df.head(3).to_string()}\n"
                )
            else:
                from pypdf import PdfReader
                text_content = ""
                if file_name.endswith('.pdf'):
                    pdf_reader = PdfReader(io.BytesIO(file_bytes))
                    for page in pdf_reader.pages:
                        text_content += (page.extract_text() or "") + "\n"
                text_var_name = f"text{idx+1}" if len(files) > 1 else "text_content"
                execution_scope[text_var_name] = text_content
                context_parts.append(
                    f"File {idx+1} ({file_name}) loaded as string '{text_var_name}'. "
                    f"Content snippet:\n{text_content[:2000]}\n"
                )

        if has_tabular:
            execution_scope["px"] = px

        context = (
            "The user uploaded the following files:\n" + "\n".join(context_parts) + "\n"
            f"User Intent: {query}.\n"
            f"Write valid executable Python code processing the provided DataFrames/strings. "
            f"Assign descriptive string summaries to 'text_insight'. "
            f"If a chart is requested, assign a Plotly Express object figure to 'fig'. IMPORTANT: Before plotting, drop NaN/missing values for the plotted columns to prevent empty charts! "
            f"Also, assign a list of exactly 3 highly relevant follow-up analytical questions to a variable named 'suggested_queries'."
        )

        generated_script = engine.generate_analysis_code(prompt_context=context)
        local_vars = engine.execute_safely(code_str=generated_script, global_vars=execution_scope)
        
        fig = local_vars.get('fig')
        analytical_text = local_vars.get('text_insight', "Data processing operation concluded successfully.")
        suggestions = local_vars.get('suggested_queries', [])
        
        chart_json = None
        if fig:
            import json
            from plotly.utils import PlotlyJSONEncoder
            chart_json = json.loads(json.dumps(fig, cls=PlotlyJSONEncoder))

        return JSONResponse({
            "status": "success",
            "insight": analytical_text,
            "chart_data": chart_json,
            "suggestions": suggestions
        })
    except Exception as e:
        return JSONResponse({"status": "error", "message": str(e)}, status_code=500)



### Cell [3.2]: Dashboard Generation Endpoint

**Explanation:**  
The `/api/dashboard` endpoint requires the AI to analyze a tabular dataset and output a structured dictionary named `dashboard_data` alongside exactly **two** Plotly Express chart objects (`fig1` and `fig2`).



In [ ]:
@app.post("/api/dashboard")
async def generate_dashboard(files: list[UploadFile] = File(...)):
    try:
        file = files[0]
        file_bytes = await file.read()
        file_name = file.filename
        if not file_name.endswith(('.csv', '.xlsx')):
            return JSONResponse({"status": "error", "message": "Dashboards currently only support Tabular datasets."})
            
        if file_name.endswith('.csv'):
            df = pd.read_csv(io.BytesIO(file_bytes))
        else:
            df = pd.read_excel(io.BytesIO(file_bytes))
            
        df = df.dropna(how='all').loc[:, ~df.columns.str.contains('^Unnamed', case=False, na=False)]
        df.columns = df.columns.str.strip()
        
        context = (
            f"Generate a comprehensive dashboard from this dataset. Columns: {list(df.columns)}.\n"
            f"Data sample:\n{df.head(3).to_string()}\n\n"
            "REQUIREMENTS:\n"
            "1. Write Python code to analyze the dataframe `df`.\n"
            "2. Assign a dictionary to `dashboard_data` with these EXACT keys:\n"
            "   - 'anomalies' (list of strings, 2 items max)\n"
            "   - 'trends' (list of strings, 2 items max)\n"
            "   - 'summary' (string paragraph overview)\n"
            "   - 'chart1_title' (string describing fig1)\n"
            "   - 'chart2_title' (string describing fig2)\n"
            "   - 'dataset_description' (string 1-2 sentence description of the dataset and quality)\n"
            "3. You MUST generate exactly TWO Plotly express figures and assign them to the variables `fig1` and `fig2`.\n"
        )
        
        import plotly.express as px
        execution_scope = {"df": df, "px": px, "pd": pd}
        generated_script = engine.generate_analysis_code(prompt_context=context)
        local_vars = engine.execute_safely(code_str=generated_script, global_vars=execution_scope)
        
        dash_data = local_vars.get('dashboard_data', {})
        fig1 = local_vars.get('fig1')
        fig2 = local_vars.get('fig2')
        
        import json
        from plotly.utils import PlotlyJSONEncoder
        
        chart1_json = json.loads(json.dumps(fig1, cls=PlotlyJSONEncoder)) if fig1 else None
        chart2_json = json.loads(json.dumps(fig2, cls=PlotlyJSONEncoder)) if fig2 else None

        return JSONResponse({
            "status": "success",
            "anomalies": dash_data.get('anomalies', []),
            "trends": dash_data.get('trends', []),
            "summary": dash_data.get('summary', "Summary not available."),
            "chart1_title": dash_data.get('chart1_title', "Primary Visualization"),
            "chart2_title": dash_data.get('chart2_title', "Secondary Visualization"),
            "dataset_description": dash_data.get('dataset_description', "Tabular dataset analysis."),
            "chart1": chart1_json,
            "chart2": chart2_json
        })
    except Exception as e:
        return JSONResponse({"status": "error", "message": str(e)}, status_code=500)



---

## 4. Frontend Client SPA (`index.html`)

The frontend is a styled Single Page Application (SPA). It uses JavaScript to toggle views dynamically, query backend endpoints, parse JSON responses, and draw custom interactive visualizations via Plotly.js.

### Cell [4.1]: View Switching & Layout Management

**Explanation:**  
Instead of reloading the browser window, `switchView` toggles CSS classes (`.hidden`) across target element containers to change views (Chat, Dashboard, or Dataset). It updates the navigation bar styling dynamically.



In [ ]:
function switchView(view) {
    viewMode = view;
    const tabChat = document.getElementById('tabChat');
    const tabDash = document.getElementById('tabDash');
    const tabData = document.getElementById('tabData');
    
    const chatCont = document.getElementById('chatContainer');
    const dashCont = document.getElementById('dashboardContainer');
    const dataCont = document.getElementById('datasetContainer');
    
    const printBtn = document.getElementById('printBtn');
    const inputAreaWrapper = document.getElementById('inputAreaWrapper');

    // Default classes styling variables
    const inactiveClass = "px-5 py-1.5 rounded-none text-xs font-bold text-slate-400 hover:text-white transition-all uppercase tracking-wide";
    const activeClass = "px-5 py-1.5 rounded-none text-xs font-bold bg-[#0F172A] text-white transition-all uppercase tracking-wide";
    
    tabChat.className = inactiveClass;
    tabDash.className = inactiveClass;
    tabData.className = inactiveClass;
    
    chatCont.classList.add('hidden');
    dashCont.classList.add('hidden');
    dataCont.classList.add('hidden');

    if (view === 'chat') {
        tabChat.className = activeClass;
        chatCont.classList.remove('hidden');
        printBtn.classList.add('hidden');
        inputAreaWrapper.classList.remove('hidden');
        document.getElementById('welcomeHeroArea').classList.remove('hidden');
    } else if (view === 'dashboard') {
        tabDash.className = activeClass;
        dashCont.classList.remove('hidden');
        printBtn.classList.remove('hidden');
        printBtn.classList.add('flex');
        inputAreaWrapper.classList.add('hidden');
        document.getElementById('welcomeHeroArea').classList.add('hidden');
    } else if (view === 'dataset') {
        tabData.className = activeClass;
        dataCont.classList.remove('hidden');
        printBtn.classList.add('hidden');
        inputAreaWrapper.classList.add('hidden');
        document.getElementById('welcomeHeroArea').classList.add('hidden');
    }
}



### Cell [4.2]: Executing Chat Analysis & Plotting Charts

**Explanation:**  
When the user submits a prompt, `submitAnalysisQuery` packages the binary file attachments and text inputs into a `FormData` envelope, sends a POST request to `/api/analyze`, parses the output strings, and runs `Plotly.newPlot()` on the client's canvas space if a figure configuration is returned in `chart_data`.



In [ ]:
async function submitAnalysisQuery() {
    let queryString = queryInput.value;
    if (activeSelectedFiles.length === 0 || !queryString.trim()) return;

    if (activeQuotedContext) {
        queryString = `[Context: ${activeQuotedContext}] \n\n` + queryString;
    }

    document.getElementById('welcomeHeroArea').classList.add('collapsed');
    chatContainer.classList.remove('hidden');
    chatContainer.classList.add('flex');

    chatContainer.insertAdjacentHTML('beforeend', createUserMessageHtml(queryInput.value));
    
    const msgId = ++messageCounter;
    chatContainer.insertAdjacentHTML('beforeend', createAiMessageHtml(msgId));
    lucide.createIcons();

    window.scrollTo({ top: document.body.scrollHeight, behavior: 'smooth' });

    const formData = new FormData();
    activeSelectedFiles.forEach(f => formData.append("files", f));
    formData.append("query", queryString);

    queryInput.value = "";
    queryInput.style.height = "auto";
    clearQuotedContext();

    try {
        const response = await fetch("http://127.0.0.1:8000/api/analyze", {
            method: "POST",
            body: formData
        });

        const data = await response.json();

        if (data.status === "success") {
            document.getElementById(`text-${msgId}`).innerText = data.insight;
            document.getElementById(`reply-btn-${msgId}`).classList.remove('hidden');
            document.getElementById(`reply-btn-${msgId}`).classList.add('flex');

            if (data.chart_data) {
                const chartDiv = document.getElementById(`chart-${msgId}`);
                const downloadBtn = document.getElementById(`download-btn-${msgId}`);
                chartDiv.style.display = 'block';
                downloadBtn.classList.remove('hidden');
                downloadBtn.classList.add('flex');
                
                Plotly.newPlot(`chart-${msgId}`, data.chart_data.data, data.chart_data.layout, plotModeBarConfig);
            }

            if (data.suggestions && data.suggestions.length > 0) {
                const sugBlock = document.getElementById('suggestionsBlock');
                const innerBtns = data.suggestions.map(s => 
                    `<button onclick="useSuggestion(this.innerText)" class="shrink-0 bg-slate-800 border border-slate-700 hover:border-slate-500 text-slate-400 hover:text-white px-4 py-2 rounded-none text-xs font-bold uppercase tracking-wide transition-colors">${s}</button>`
                ).join('');
                
                sugBlock.innerHTML = innerBtns + `
                    <button type="button" onclick="dismissSuggestions()" class="shrink-0 flex items-center justify-center w-8 h-8 bg-slate-800 hover:bg-rose-900 text-slate-500 hover:text-rose-400 border border-slate-700 hover:border-rose-800 rounded-none transition-colors ml-auto">
                        <i data-lucide="x" class="w-4 h-4"></i>
                    </button>
                `;
                
                document.getElementById('sugToggleBtn').classList.add('hidden');
                sugBlock.classList.remove('hidden');
                sugBlock.classList.add('flex');
            }
        } else {
            document.getElementById(`text-${msgId}`).innerHTML = `<span class="text-sm font-bold uppercase tracking-wider text-rose-600">Exception: ${data.message}</span>`;
        }
    } catch (err) {
        document.getElementById(`text-${msgId}`).innerHTML = `<span class="text-sm font-bold uppercase tracking-wider text-rose-600">Request Failed: ${err.message || 'Connection Terminated'}</span>`;
    }
    
    lucide.createIcons();
    window.scrollTo({ top: document.body.scrollHeight, behavior: 'smooth' });
}



### Cell [4.3]: Dashboard Ingestion & Visualization Setup

**Explanation:**  
Triggers when generating or updating dashboards. It inserts dynamic cards and mounts two initial Plotly express figures.



In [ ]:
async function generateDashboard() {
    if(activeSelectedFiles.length === 0) return alert("Please select a file first.");
    
    document.getElementById('dashContent').classList.add('hidden');
    document.getElementById('dashContent').classList.remove('flex');
    document.getElementById('dashLoader').classList.remove('hidden');
    document.getElementById('dashLoader').classList.add('flex');

    const formData = new FormData();
    activeSelectedFiles.forEach(f => formData.append("files", f));

    try {
        const response = await fetch("http://127.0.0.1:8000/api/dashboard", { method: "POST", body: formData });
        const data = await response.json();

        if(data.status === "success") {
            document.getElementById('dashLoader').classList.add('hidden');
            document.getElementById('dashLoader').classList.remove('flex');
            document.getElementById('dashContent').classList.remove('hidden');
            document.getElementById('dashContent').classList.add('flex');

            // Populate initial Insights Cards
            const insightsGrid = document.getElementById('dashInsightsGrid');
            insightsGrid.innerHTML = `
                ${createDashTextCardHtml('base-1', 'Anomalies', data.anomalies.length > 0 ? data.anomalies.map(a => '• ' + a).join('\\n') : 'No critical anomalies detected.')}
                ${createDashTextCardHtml('base-2', 'Key Trends', data.trends.length > 0 ? data.trends.map(t => '• ' + t).join('\\n') : 'No significant trends observed.')}
                ${createDashTextCardHtml('base-3', 'Managerial Summary', data.summary)}
            `;
            
            // Adjust card color highlights
            document.getElementById('dash-item-base-1').className = "relative group bg-transparent border-l-2 border-l-rose-500 pl-5 flex flex-col gap-3 h-full overflow-y-auto max-h-[300px]";
            document.getElementById('dash-item-base-2').className = "relative group bg-transparent border-l-2 border-l-emerald-500 pl-5 flex flex-col gap-3 h-full overflow-y-auto max-h-[300px]";
            document.getElementById('dash-item-base-3').className = "relative group bg-transparent border-l-2 border-l-blue-500 pl-5 flex flex-col gap-3 h-full overflow-y-auto max-h-[300px]";

            document.getElementById('dashDatasetName').innerText = activeSelectedFiles[0].name;

            // Render double charts grid columns
            const chartsGrid = document.getElementById('dashChartsGrid');
            chartsGrid.innerHTML = `
                ${createDashChartCardHtml('1', data.chart1_title || 'Primary Visualization')}
                ${createDashChartCardHtml('2', data.chart2_title || 'Secondary Visualization')}
            `;

            if(data.chart1) {
                Plotly.newPlot('dashChart1', data.chart1.data, data.chart1.layout, plotModeBarConfig);
            }
            if(data.chart2) {
                Plotly.newPlot('dashChart2', data.chart2.data, data.chart2.layout, plotModeBarConfig);
            }

            lucide.createIcons();
        } else {
            alert("Dashboard generation failed: " + data.message);
            document.getElementById('dashLoader').classList.add('hidden');
        }
    } catch(e) {
        alert("Service connectivity interrupted.");
        document.getElementById('dashLoader').classList.add('hidden');
    }
}

function useSuggestion(text) {
    if(activeSelectedFiles.length === 0) return alert("Please upload a file first.");
    queryInput.value = text;
    submitAnalysisQuery(); 
}
